In [1]:
import os, sys, json, time, shutil, warnings, logging
from pathlib import Path
PROJECT = Path(r"C:\Users\hp\Documents\teliport_prototype")
os.chdir(PROJECT / "notebooks")
sys.path.insert(0, str(PROJECT / "src"))
%load_ext autoreload
%autoreload 2

os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)
import numpy as np, pandas as pd, mlflow
from cmapss_data import load_all, SUBSETS

DATA_DIR = PROJECT / "data" / "raw" / "CMAPSS"
INCOMING = PROJECT / "data" / "incoming"
MODELS = PROJECT / "models"
REPORTS = PROJECT / "reports"
WO_DIR = PROJECT / "work_orders"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
data = load_all(DATA_DIR)
print("Working dir:", Path.cwd())

Working dir: C:\Users\hp\Documents\teliport_prototype\notebooks


In [2]:
try:
    (PROJECT / "mlflow.db").unlink(missing_ok=True)
    shutil.rmtree(PROJECT / "mlruns", ignore_errors=True)
    for f in MODELS.glob("deployed_*.json"):
        f.unlink()
    print("Old MLflow tracking data removed - starting a clean registry.")
except PermissionError:
    print("mlflow.db is locked: stop the MLflow UI (Ctrl + C in its PowerShell window) and run this cell again.")

Old MLflow tracking data removed - starting a clean registry.


In [3]:
%%writefile ../src/training.py
"""ONE training recipe used everywhere: notebooks, MLOps retraining and the end-to-end pipeline."""
from __future__ import annotations

import json
from pathlib import Path

import joblib

from anomaly import HealthMonitor
from features import CMAPSSPreprocessor, DEFAULT_RUL_CAP
from modeling import RULUncertainty, SeedBag, cross_validate, fit_model, make_lgbm

DEFAULT_MONITOR = {"method": "mahalanobis", "quantile": 0.99, "persistence": 5}


def load_config(models_dir, filename, fd, default):
    path = Path(models_dir) / filename if models_dir else None
    if path and path.exists():
        return json.loads(path.read_text()).get(fd, default)
    return default


def train_pipeline(fd, train_raw, models_dir=None, params=None, monitor_cfg=None,
                   cap=DEFAULT_RUL_CAP, seeds=(0, 1, 2, 3, 4), calib_folds=3):
    """train_raw: run-to-failure history with an RUL column. Returns every fitted component."""
    params = params if params is not None else load_config(models_dir, "best_params.json", fd, {})
    monitor_cfg = monitor_cfg or load_config(models_dir, "monitor_config.json", fd, DEFAULT_MONITOR)

    prep = CMAPSSPreprocessor.for_subset(fd)
    tr = prep.fit_transform(train_raw)
    F = prep.feature_names_

    _, oof = cross_validate(tr, F, lambda: make_lgbm(**params), cap, n_splits=calib_folds)
    unc = RULUncertainty().fit(oof["y_true"], oof["y_pred"])          # calibrated on unseen engines
    model = fit_model(tr, F, lambda: SeedBag(params, seeds), cap, weighted=True)
    hm = HealthMonitor(**monitor_cfg).fit(tr, prep.sensors)
    return {"prep": prep, "model": model, "unc": unc, "hm": hm, "features": F,
            "params": params, "monitor_cfg": monitor_cfg, "train_features": tr}


def save_pipeline(p, fd, models_dir):
    models_dir = Path(models_dir)
    models_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(p["prep"], models_dir / f"preprocessor_{fd}.joblib")
    joblib.dump(p["model"], models_dir / f"lgbm_{fd}.joblib")
    joblib.dump(p["unc"], models_dir / f"uncertainty_{fd}.joblib")
    joblib.dump(p["hm"], models_dir / f"health_monitor_{fd}.joblib")
    stale = models_dir / f"deployed_{fd}.json"          # MLOps deploy() rewrites it right after
    if stale.exists():
        stale.unlink()

Overwriting ../src/training.py


In [4]:
%%writefile ../src/serving.py
"""MLflow pyfunc wrapper: the registered model is the WHOLE pipeline (raw sensors in -> RUL range out)."""
from __future__ import annotations

import numpy as np
import pandas as pd
import mlflow.pyfunc


class RULPipelineModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import joblib
        self.prep = joblib.load(context.artifacts["preprocessor"])
        self.model = joblib.load(context.artifacts["rul_model"])
        self.unc = joblib.load(context.artifacts["uncertainty"])

    def predict(self, context, model_input, params=None):
        from features import last_cycle
        last = last_cycle(self.prep.transform(model_input))
        pred = np.clip(self.model.predict(last[self.prep.feature_names_]), 0, None)
        lower, upper = self.unc.interval(pred)
        return pd.DataFrame({"unit": last["unit"].to_numpy(), "pred_RUL": np.round(pred, 1),
                             "rul_lower": np.round(lower, 1), "rul_upper": np.round(upper, 1)})

Writing ../src/serving.py


In [4]:
%%writefile ../src/mlops.py
"""MLOps: MLflow experiment tracking, model registry, drift check and gated retraining flow.

Promotion decisions use a fixed GATE set of training engines that no model ever trains on.
The NASA test set is logged for reporting only and never drives a decision.
"""
from __future__ import annotations

import json
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import joblib
import mlflow
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

from anomaly import detection_report
from cmapss_data import COLUMNS, OP_COLS, SENSOR_COLS
from features import last_cycle
from modeling import evaluate, val_rows
from serving import RULPipelineModel
from training import save_pipeline, train_pipeline

EXPERIMENT = "teliport-predictive-maintenance"
GATE_FRACTION = 0.15
SRC_FILES = ["cmapss_data.py", "features.py", "modeling.py", "anomaly.py", "serving.py"]


def clean_key(k):
    """MLflow metric names cannot contain '<=' or '%'."""
    return k.replace("<=", "_le_").replace("%", "pct")


def model_name(fd):
    return f"teliport_rul_{fd}"


def setup_mlflow(project):
    project = Path(project)
    mlflow.set_tracking_uri(f"sqlite:///{(project / 'mlflow.db').as_posix()}")
    if mlflow.get_experiment_by_name(EXPERIMENT) is None:
        mlflow.create_experiment(EXPERIMENT, artifact_location=(project / "mlruns").as_uri())
    mlflow.set_experiment(EXPERIMENT)
    return MlflowClient()


def split_gate(train_raw, fraction=GATE_FRACTION, seed=42):
    """Fixed, reproducible set of engines kept out of ALL training -> fair champion/challenger gate."""
    units = np.array(sorted(train_raw["unit"].unique()))
    gate = np.random.default_rng(seed).choice(units, size=max(1, round(fraction * len(units))), replace=False)
    is_gate = train_raw["unit"].isin(gate)
    return train_raw[~is_gate].copy(), train_raw[is_gate].copy()


# ---------------- data checks ----------------
def validate_batch(df):
    """Schema + sanity checks before new data is allowed into training."""
    issues = []
    missing = [c for c in COLUMNS if c not in df.columns]
    if missing:
        issues.append(f"missing columns: {missing}")
    else:
        if df[SENSOR_COLS + OP_COLS].isna().mean().max() > 0.2:
            issues.append("more than 20% missing values in a column")
        if df.duplicated(["unit", "cycle"]).any():
            issues.append(f"{int(df.duplicated(['unit', 'cycle']).sum())} duplicate (unit, cycle) rows (will be de-duplicated)")
        if (df["cycle"] < 1).any():
            issues.append("cycle numbers < 1")
    blocking = [i for i in issues if "duplicate" not in i]
    return {"ok": not blocking, "issues": issues, "rows": len(df), "engines": int(df["unit"].nunique())}


def psi(ref, new, bins=10):
    """Population Stability Index: <0.1 stable, 0.1-0.25 moderate shift, >0.25 major shift."""
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref) + 1e-6
    n = np.histogram(new, edges)[0] / len(new) + 1e-6
    return float(np.sum((n - r) * np.log(n / r)))


def drift_report(prep, ref_raw, new_raw):
    ref = prep.transform(ref_raw)
    new = prep.transform(new_raw)
    rep = pd.Series({s: psi(ref[f"{s}_z"], new[f"{s}_z"]) for s in prep.sensors}).round(3)
    return rep.sort_values(ascending=False)


# ---------------- evaluation ----------------
def evaluate_on_gate(p, gate_raw):
    feats = val_rows(p["prep"].transform(gate_raw))
    pred = np.clip(p["model"].predict(feats[p["features"]]), 0, None)
    lower, upper = p["unc"].interval(pred)
    y = feats["RUL"].to_numpy()
    m = {clean_key(k): v for k, v in evaluate(y, pred).items() if k != "n"}
    m["interval_coverage_pct"] = round(100 * float(np.mean((y >= lower) & (y <= upper))), 1)
    return m


def evaluate_on_test(p, test_raw, test_rul):
    te = last_cycle(p["prep"].transform(test_raw)).merge(test_rul[["unit", "RUL"]], on="unit")
    pred = np.clip(p["model"].predict(te[p["features"]]), 0, None)
    return {clean_key(k): v for k, v in evaluate(te["RUL"], pred).items() if k != "n"}


def _offline_predict(p, raw):
    last = last_cycle(p["prep"].transform(raw))
    pred = np.clip(p["model"].predict(last[p["features"]]), 0, None)
    lower, upper = p["unc"].interval(pred)
    return pd.DataFrame({"unit": last["unit"].to_numpy(), "pred_RUL": np.round(pred, 1),
                         "rul_lower": np.round(lower, 1), "rul_upper": np.round(upper, 1)})


# ---------------- train + log + register ----------------
def train_and_log(fd, train_raw, gate_raw, test_raw, test_rul, models_dir, run_name, tags=None, extra_params=None):
    p = train_pipeline(fd, train_raw, models_dir=models_dir)
    gate_m = evaluate_on_gate(p, gate_raw)
    test_m = evaluate_on_test(p, test_raw, test_rul)
    det = detection_report(p["train_features"], p["hm"].transform(p["train_features"]))

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({"subset": fd, **(tags or {})})
        mlflow.log_params({"subset": fd, "train_engines": int(train_raw["unit"].nunique()),
                           "gate_engines": int(gate_raw["unit"].nunique()), "train_rows": len(train_raw),
                           "n_features": len(p["features"]), "n_seeds": len(p["model"].seeds),
                           **{f"lgb_{k}": v for k, v in p["params"].items()},
                           **{f"monitor_{k}": v for k, v in p["monitor_cfg"].items()}, **(extra_params or {})})
        mlflow.log_metrics({f"gate_{k}": float(v) for k, v in gate_m.items()})
        mlflow.log_metrics({f"test_{k}": float(v) for k, v in test_m.items()})
        mlflow.log_metric("anomaly_false_alarm_pct", float(det["false_alarm_rate_%"]))
        with tempfile.TemporaryDirectory() as tmp:
            tmp = Path(tmp)
            paths = {"preprocessor": tmp / "preprocessor.joblib", "rul_model": tmp / "rul_model.joblib",
                     "uncertainty": tmp / "uncertainty.joblib", "health_monitor": tmp / "health_monitor.joblib"}
            for key, obj in [("preprocessor", p["prep"]), ("rul_model", p["model"]),
                             ("uncertainty", p["unc"]), ("health_monitor", p["hm"])]:
                joblib.dump(obj, paths[key])
            src = Path(__file__).resolve().parent
            example = test_raw[test_raw["unit"].isin(test_raw["unit"].unique()[:3])]
            info = mlflow.pyfunc.log_model(
                name="rul_pipeline", python_model=RULPipelineModel(),
                artifacts={k: str(v) for k, v in paths.items()},
                code_paths=[str(src / f) for f in SRC_FILES],
                signature=infer_signature(example, _offline_predict(p, example)),
                registered_model_name=model_name(fd))
        version = info.registered_model_version
    return {"run_id": run.info.run_id, "version": str(version), "gate": gate_m, "test": test_m, "pipeline": p}


def set_champion(client, fd, version):
    client.set_registered_model_alias(model_name(fd), "champion", str(version))


def get_champion(client, fd):
    try:
        mv = client.get_model_version_by_alias(model_name(fd), "champion")
    except Exception:
        return None
    return {"version": str(mv.version), "run_id": mv.run_id, "metrics": client.get_run(mv.run_id).data.metrics}


def deploy(result, fd, models_dir):
    """Copy the champion pipeline to the folder that inference / dashboard / agent read from."""
    save_pipeline(result["pipeline"], fd, models_dir)
    (Path(models_dir) / f"deployed_{fd}.json").write_text(json.dumps(
        {"subset": fd, "version": result["version"], "run_id": result["run_id"],
         "deployed_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}, indent=2))


# ---------------- retraining flow ----------------
def retrain_on_new_data(fd, current_train_raw, gate_raw, new_batch_raw, test_raw, test_rul, client,
                        models_dir, max_rmse_increase=0.0, max_critical_rmse_increase=0.5):
    """New run-to-failure data arrived -> validate -> drift check -> retrain challenger ->
    compare with champion on the GATE engines -> promote + deploy only if not worse."""
    log = {"subset": fd, "started_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    v = validate_batch(new_batch_raw)
    log["validation"] = v
    if not v["ok"]:
        log["decision"] = "REJECTED_BAD_DATA"
        return log

    new = new_batch_raw.copy()
    new["unit"] = new["unit"] + int(max(current_train_raw["unit"].max(), gate_raw["unit"].max()))
    new["RUL"] = new.groupby("unit")["cycle"].transform("max") - new["cycle"]
    combined = pd.concat([current_train_raw, new], ignore_index=True)

    champ = get_champion(client, fd)
    drift = drift_report(joblib.load(Path(models_dir) / f"preprocessor_{fd}.joblib"), current_train_raw, new)
    log["drift_max_psi"] = float(drift.max())
    log["drift_top"] = drift.head(3).to_dict()

    res = train_and_log(fd, combined, gate_raw, test_raw, test_rul, models_dir,
                        run_name=f"{fd}-retrain-{datetime.now():%Y%m%d-%H%M%S}",
                        tags={"trigger": "new_data", "stage": "challenger"},
                        extra_params={"new_engines": v["engines"], "drift_max_psi": round(float(drift.max()), 3)})
    ch = res["gate"]
    log["challenger"] = {"version": res["version"], "gate_RMSE": ch["RMSE"], "gate_RMSE_RUL_le_30": ch["RMSE_RUL_le_30"]}

    if champ is None:
        better, reason = True, "no champion yet"
    else:
        c_rmse, c_crit = champ["metrics"]["gate_RMSE"], champ["metrics"]["gate_RMSE_RUL_le_30"]
        log["champion"] = {"version": champ["version"], "gate_RMSE": c_rmse, "gate_RMSE_RUL_le_30": c_crit}
        better = (ch["RMSE"] <= c_rmse + max_rmse_increase) and \
                 (ch["RMSE_RUL_le_30"] <= c_crit + max_critical_rmse_increase)
        reason = (f"gate RMSE {ch['RMSE']} vs champion {c_rmse}; "
                  f"gate critical-zone RMSE {ch['RMSE_RUL_le_30']} vs {c_crit}")

    with mlflow.start_run(run_id=res["run_id"]):
        mlflow.set_tag("promotion_decision", "PROMOTED" if better else "KEPT_CHAMPION")
        mlflow.log_dict({k: val for k, val in log.items() if k != "validation"}, "retraining_log.json")

    if better:
        set_champion(client, fd, res["version"])
        deploy(res, fd, models_dir)
        log["decision"] = f"PROMOTED v{res['version']} to champion and deployed ({reason})"
    else:
        log["decision"] = f"KEPT champion v{champ['version']} ({reason})"
    log["test_RMSE_report_only"] = res["test"]["RMSE"]
    return log

Overwriting ../src/mlops.py


In [5]:
# --- clean restart (MLflow UI must be closed) ---
(PROJECT / "mlflow.db").unlink(missing_ok=True)
shutil.rmtree(PROJECT / "mlruns", ignore_errors=True)
for f in MODELS.glob("deployed_*.json"):
    f.unlink()
print("Old MLflow data removed - clean registry.\n")

# --- Cell 6: version 1 for all subsets (train -> gate evaluate -> log -> register -> deploy) ---
from mlops import (setup_mlflow, train_and_log, set_champion, deploy, retrain_on_new_data,
                   split_gate, EXPERIMENT, model_name)

client = setup_mlflow(PROJECT)
splits, rows = {}, []
for fd in SUBSETS:
    train, test, rul = data[fd]
    train = train.drop(columns=["subset"])
    pool = train[train["unit"] <= 80] if fd == "FD001" else train     # FD001 demo: 20 engines arrive later
    tr_part, gate = split_gate(pool)
    splits[fd] = (tr_part, gate)
    res = train_and_log(fd, tr_part, gate, test, rul, MODELS, run_name=f"{fd}-initial",
                        tags={"trigger": "initial", "stage": "champion", "promotion_decision": "INITIAL_CHAMPION"})
    set_champion(client, fd, res["version"])
    deploy(res, fd, MODELS)
    rows.append({"subset": fd, "version": res["version"], "train_engines": tr_part["unit"].nunique(),
                 "gate_engines": gate["unit"].nunique(), "gate_RMSE": res["gate"]["RMSE"],
                 "gate_RMSE_RUL<=30": res["gate"]["RMSE_RUL_le_30"],
                 "gate_coverage_%": res["gate"]["interval_coverage_pct"],
                 "test_RMSE (report only)": res["test"]["RMSE"]})
    print(fd, "logged + registered + deployed", flush=True)
pd.DataFrame(rows).set_index("subset")

Old MLflow data removed - clean registry.



C:\Users\hp\Documents\teliport_prototype\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/10/30 22:39:13 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/30 22:39:13 INFO mlflow.store.db.utils: Updating database tables
2026/10/30 22:39:25 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


FD001 logged + registered + deployed


Successfully registered model 'teliport_rul_FD001'.
Created version '1' of model 'teliport_rul_FD001'.
2026/10/30 22:39:52 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


FD002 logged + registered + deployed


Successfully registered model 'teliport_rul_FD002'.
Created version '1' of model 'teliport_rul_FD002'.
2026/10/30 22:40:11 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
Successfully registered model 'teliport_rul_FD003'.
Created version '1' of model 'teliport_rul_FD003'.


FD003 logged + registered + deployed


2026/10/30 22:40:38 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
Successfully registered model 'teliport_rul_FD004'.
Created version '1' of model 'teliport_rul_FD004'.


FD004 logged + registered + deployed


,version,train_engines,gate_engines,gate_RMSE,gate_RMSE_RUL<=30,gate_coverage_%,test_RMSE (report only)
subset,,,,,,,
FD001,1,68,12,13.52,2.87,92.2,11.74
FD002,1,221,39,13.51,3.61,90.0,23.09
FD003,1,85,15,11.63,3.58,90.9,12.34
FD004,1,212,37,14.05,8.19,90.8,24.22


In [6]:
INCOMING.mkdir(parents=True, exist_ok=True)
train, test, rul = data["FD001"]
batch = train[train["unit"] > 80].drop(columns=["RUL", "subset"]).copy()
batch["unit"] = batch["unit"] - 80                      # new fleet numbers its engines from 1
batch.to_csv(INCOMING / "FD001_batch_001.csv", index=False)

new_batch = pd.read_csv(INCOMING / "FD001_batch_001.csv")
tr_part, gate = splits["FD001"]
t0 = time.time()
log = retrain_on_new_data("FD001", tr_part, gate, new_batch, test, rul, client, MODELS)
print(f"Retraining flow finished in {time.time() - t0:.1f}s\n")
print(json.dumps(log, indent=2, default=str))

2026/10/30 22:42:07 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
Registered model 'teliport_rul_FD001' already exists. Creating a new version of this model...
Created version '2' of model 'teliport_rul_FD001'.


Retraining flow finished in 14.9s

{
  "subset": "FD001",
  "started_at": "2026-10-30T17:11:57+00:00",
  "validation": {
    "ok": true,
    "issues": [],
    "rows": 4493,
    "engines": 20
  },
  "drift_max_psi": 0.083,
  "drift_top": {
    "s14": 0.083,
    "s9": 0.051,
    "s13": 0.021
  },
  "challenger": {
    "version": "2",
    "gate_RMSE": 12.73,
    "gate_RMSE_RUL_le_30": 2.98
  },
  "champion": {
    "version": "1",
    "gate_RMSE": 13.52,
    "gate_RMSE_RUL_le_30": 2.87
  },
  "decision": "PROMOTED v2 to champion and deployed (gate RMSE 12.73 vs champion 13.52; gate critical-zone RMSE 2.98 vs 2.87)",
  "test_RMSE_report_only": 11.2
}


In [7]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=["attributes.start_time ASC"])
cols = {"tags.mlflow.runName": "run", "params.subset": "subset", "params.train_engines": "train_engines",
        "params.gate_engines": "gate_engines", "metrics.gate_RMSE": "gate_RMSE",
        "metrics.gate_RMSE_RUL_le_30": "gate_RMSE_RUL<=30", "metrics.gate_interval_coverage_pct": "gate_coverage_%",
        "metrics.test_RMSE": "test_RMSE (report only)", "tags.promotion_decision": "decision"}
history = runs[[c for c in cols if c in runs.columns]].rename(columns=cols)
history.to_csv(REPORTS / "mlflow_run_history.csv", index=False)
display(history)

for fd in SUBSETS:
    mv = client.get_model_version_by_alias(model_name(fd), "champion")
    print(f"{fd}: champion = v{mv.version}")

# the registered model is the full pipeline: raw sensor history in -> RUL range out
champion = mlflow.pyfunc.load_model(f"models:/{model_name('FD001')}@champion")
_, test, rul = data["FD001"]
champion.predict(test[test["unit"].isin([1, 2, 3])]).merge(rul[["unit", "RUL"]], on="unit")

,run,subset,train_engines,gate_engines,gate_RMSE,gate_RMSE_RUL<=30,gate_coverage_%,test_RMSE (report only),decision
0,FD001-initial,FD001,68,12,13.52,2.87,92.2,11.74,INITIAL_CHAMPION
1,FD002-initial,FD002,221,39,13.51,3.61,90.0,23.09,INITIAL_CHAMPION
2,FD003-initial,FD003,85,15,11.63,3.58,90.9,12.34,INITIAL_CHAMPION
3,FD004-initial,FD004,212,37,14.05,8.19,90.8,24.22,INITIAL_CHAMPION
4,FD001-retrain-20261030-224157,FD001,88,12,12.73,2.98,91.9,11.20,PROMOTED


FD001: champion = v2
FD002: champion = v1
FD003: champion = v1
FD004: champion = v1


,unit,pred_RUL,rul_lower,rul_upper,RUL
0,1,119.3,94.3,149.3,112
1,2,116.8,91.8,146.8,98
2,3,48.7,35.4,60.7,69


In [8]:
from inference import score_fleet
from triage_agent import TriageAgent, TriageConfig

snaps = pd.concat([score_fleet(data[fd][1], fd, MODELS) for fd in SUBSETS], ignore_index=True)
shutil.rmtree(WO_DIR, ignore_errors=True)
decisions = TriageAgent(TriageConfig(conservative_mode="critical"), out_dir=WO_DIR).run(snaps)
decisions.to_csv(REPORTS / "triage_decisions_test.csv", index=False)
print("Model versions in use:", snaps.groupby("subset")["model_version"].first().to_dict())
print("Work orders written  :", len(list(WO_DIR.glob("WO-*.json"))))
decisions["wo_status"].value_counts()

Model versions in use: {'FD001': 'registry v2', 'FD002': 'registry v1', 'FD003': 'registry v1', 'FD004': 'registry v1'}
Work orders written  : 333


wo_status
NO_ACTION               374
AUTO_APPROVED           327
PENDING_HUMAN_REVIEW      6
Name: count, dtype: int64